# TODO: manuscript title

Ondřej Mottl [](https://orcid.org/0000-0002-9796-5081) (Department of Botany, Faculty of Science, Charles University

)  
TODO Co-author 2 [](https://orcid.org/0000-0000-0000-0000) (TODO Institution)  
TODO Co-author 3 [](https://orcid.org/0000-0000-0000-0000) (TODO Institution)  
April 27, 2026

TODO: Write abstract (~150 words for Nature Letter). Background sentence(s). Key question. Approach in one sentence. Key findings (2–3 sentences). Broader implication (1 sentence).

In [ ]:
library(here)

here() starts at D:/GITHUB/BIODYNAMICS_vegetation_cooccurrence.worktrees/main/Documentation/Manuscript

here() starts at D:/GITHUB/BIODYNAMICS_vegetation_cooccurrence.worktrees/main

- The library is already synchronized with the lockfile.

## 0.1 Introduction

TODO: Write introduction.

## 0.2 Methods

In [ ]:
here::i_am("Documentation/Manuscript/sections/03_methods.qmd")

here() starts at D:/GITHUB/BIODYNAMICS_vegetation_cooccurrence.worktrees/main

### 0.2.1 Vegetation and palaeoclimate data

We sourced all community and climate records from **VegVault** (version 1.0.0; \[cite\]), a harmonised SQLite database that integrates fossil pollen archives with co-registered palaeoclimate reconstructions and plant functional traits. Sourcing all three data types required by this study — vegetation community records, co-registered palaeoclimate predictors, and plant functional traits — from a single harmonised repository offers two critical advantages. First, it guarantees consistent site identifiers and temporal grids across all inputs, which is a prerequisite for the aligned multi-species modelling framework used here. Second, because VegVault is versioned and publicly archived, the exact set of records used in this study can be retrieved by any researcher with access to the same database version, making the data inputs fully reproducible. Fossil pollen records are drawn from the Neotoma Paleoecology Database ([**Williams2018?**](#ref-Williams2018)) and FOSSILPOL \[cite\]; plant functional traits are drawn from TRY ([**Kattge2020?**](#ref-Kattge2020)). Palaeoclimate predictors are CHELSA-TraCE21k bioclimatic variables ([**Karger2021?**](#ref-Karger2021)) co-registered in VegVault at the site and time-step level.

We selected seven CHELSA bioclimatic variables as candidate abiotic predictors: BIO1 (mean annual temperature), BIO4 (temperature seasonality), BIO6 (minimum temperature of the coldest month), BIO12 (annual precipitation), BIO15 (precipitation seasonality), BIO18 (precipitation of the warmest quarter), and BIO19 (precipitation of the coldest quarter). These variables represent the primary dimensions of climate space known to govern plant distributions — mean thermal energy, climatic extremes, and water availability across seasons — while omitting correlated secondary variables to limit redundancy before formal collinearity screening. Full descriptions and units of all bioclimatic variables are provided in Supplementary Table S\[TODO\].

The geographic and temporal coverage of extracted records (number of fossil pollen sites per 500-year time slice and per continental region) is summarised in Supplementary Figure S\[TODO\].

### 0.2.2 Taxonomic harmonisation

Fossil pollen records originate from multiple source databases that use different original nomenclature systems and taxonomic conventions; a unified classification is therefore a prerequisite for any cross-database comparison. The guiding principle of the harmonisation was to resolve each pollen morphotype to the **finest taxonomic rank the morphological evidence supports**, rather than imposing a fixed target rank across all morphotypes. For each morphotype, we first attempted to assign a genus-level identification against the GBIF taxonomic backbone \[cite\], which provides a globally consistent, community-maintained reference taxonomy. Where the morphotype was too broadly defined morphologically to support a confident genus assignment — typically for compound types that aggregate morphologically similar pollen from several genera — the record was retained at family rank rather than discarded, preserving ecological information that would otherwise be lost entirely. We excluded non-plant records (e.g., fungal spores and fern spores flagged in source databases). We supplemented the backbone matching with a curated project-specific correction table that records manually validated taxonomic decisions for morphotypes known to be mis-classified or absent from the backbone, ensuring that recurring edge cases are resolved consistently across all analyses. All classifications were reviewed and validated by expert botanists to catch systematic errors arising from regional nomenclatural traditions or legacy morphotype names not covered by the backbone. Because the taxa present in any given analysis depend on the geographic extent and temporal window queried, we compiled a separate harmonisation table for each continental region rather than a single global table. We provide these tables as Supplementary Tables S\[TODO\] (one per continental region).

### 0.2.3 Multi-scale analytical framework

Co-occurrence patterns are shaped by processes operating at different spatial extents, across geological time, and at different levels of biological organisation. Because the relative importance of these processes is itself expected to vary across scales, pooling all data at a single arbitrary scale risks masking or confounding mechanisms that are only visible at specific extents or resolutions. We therefore applied the same modelling and analysis workflow at multiple levels along three axes: **spatial** (continental, regional, and local extents), **temporal** (40 successive 500-year slices from the Last Glacial Maximum to the present), and **taxonomic** (genus, family, and functional-type level). We first describe the shared pre-processing and modelling steps common to all analyses, then detail how each axis was defined and the reasoning behind specific choices.

#### 0.2.3.1 Shared pre-processing

**Community data preparation.** We applied several pre-processing steps to the raw community matrices to prepare them for modelling and ensure comparability across analyses. We converted raw pollen counts to proportional abundance (relative pollen percentage). To standardise temporal resolution across irregular sampling intervals (typical for palaeoecological datasets), we then linearly interpolated records to a common 500-year temporal grid. We considered binning but preferred interpolation to preserve the full temporal resolution of the data and avoid artefacts introduced by arbitrary bin edges. We excluded taxa contributing less than 1 % of the pollen sum within any individual sample from that sample’s record, because such occurrences are below the detection limit reliably resolvable from pollen counts and contribute predominantly noise to co-occurrence estimates. We used presence/absence rather than continuous abundance as the response variable, because pollen abundances are strongly influenced by taxon-specific pollen production rates and dispersal distances — biases that are effectively removed by binarisation, leaving a response that more directly reflects co-occurrence \[TODO: cite\]. Finally, to ensure sufficient data density for reliable model estimates, we excluded spatial units with too few sites or too few qualifying taxa after filtering. We calibrated the exact thresholds separately for each spatial unit, because the number of sites and taxa varies across units and the optimal thresholds depend on the data structure of each unit.

**Abiotic predictor selection.** Collinear predictors inflate the variance of coefficient estimates and may result in overfitting, because variance attributable to one predictor is spuriously shared with its collinear counterparts. We therefore screened candidate bioclimatic variables for redundancy using a dual-filtering approach that combines pairwise Pearson correlation and Variance Inflation Factor (VIF) \[cite Benito 2025\]. We calibrated both thresholds automatically to the correlation structure of each specific dataset rather than setting fixed global values, because optimal thresholds depend on the number and collinearity of predictors, which vary across spatial and temporal units. When two predictors were redundant, we preferentially retained the one with greater univariate predictive power, ensuring that the non-redundant subset maximises ecological information.

**Variable scaling.** We centred and z-score scaled bioclimatic predictors to a common variance, which is required for stable gradient-descent optimisation and allows coefficient magnitudes to be compared across predictors with different natural units. In analyses with age as a predictor, we centred age but did not variance-scale it, so that interaction coefficients retain their natural interpretation in units of years. We projected site coordinates to a metric coordinate system (ETRS89-LAEA, EPSG:3035) so that geographic distances are measured in kilometres rather than decimal degrees.

**Moran Eigenvector Maps (MEMs).** Spatial autocorrelation in species distributions arises from unmeasured environmental gradients, dispersal limitation, and historical contingency — processes that are not captured by the bioclimatic predictors but would inflate the biotic residual (B fraction) of the variance decomposition if left unaccounted for. We represented large-scale spatial structure using Moran Eigenvector Maps (MEMs), which are eigenvectors of a spatial weights matrix and span the full range of positive spatial autocorrelation present in the data \[cite\]. Including MEMs as predictor variables in the spatial component of the model approximates a spatial random field without the computational cost of full Gaussian process random effects, which would be prohibitive for the large community matrices. For purely spatial analyses, we derived MEMs from a two-dimensional geographic distance matrix among sites. For spatiotemporal analyses, we used a three-dimensional distance matrix (combining longitude, latitude, and age) instead, so that the eigenvectors capture joint spatiotemporal structure rather than spatial structure alone. We reduced the number of retained eigenvectors at finer spatial scales, reflecting the lower structural complexity of smaller analysis units.

#### 0.2.3.2 Shared modelling workflow

**Joint species distribution model.** We modelled plant co-occurrence using a Joint Species Distribution Model (jSDM) ([**Warton2015?**](#ref-Warton2015)), which fits all species simultaneously within a single likelihood framework and estimates the residual covariance among species after accounting for shared environmental and spatial responses. We chose the jSDM framework over single-species distribution models because it explicitly models the residual inter-specific covariance — that is, co-occurrences that cannot be attributed to shared responses to environment or space — which is the central quantity of interest in this study. We performed all analyses using the R package {sjSDM} ([**Pichler2021?**](#ref-Pichler2021)), which fits jSDMs via stochastic gradient descent with an optional GPU backend, making it feasible for the large community matrices (up to several hundred taxa × thousands of sites) analysed here.

Each taxon’s occurrence was modelled as the linear combination of three additive components:

-   **Environment (A)** — the non-collinear bioclimatic predictors, with age × climate interaction terms to allow the strength of climate filtering to vary across the Last Glacial–Interglacial cycle. only the interaction of age with climate predictors was included, the direct age term was excluded to avoid absorbing broad temporal trend into the environment component.
-   **Space (S)** — MEM predictors included as fixed effects to absorb spatially (or spatiotemporally) structured variation not explained by the climate predictors, thereby preventing unmeasured spatial processes from inflating the biotic residual.
-   **Biotic residual (B)** — a shared low-rank latent covariance matrix that captures the residual pairwise associations among taxa after accounting for their shared responses to environment and space. Positive residual covariance indicates that two taxa co-occur more often than expected given environment and spatial structure; negative covariance indicates mutual exclusion.

**Model fit and convergence.** Because the primary goal of this study is variance decomposition rather than out-of-sample prediction, we assessed model performance on the training data using Nagelkerke R² and per-species AUC, accuracy, and log-loss. Convergence was confirmed by inspecting the training loss trajectory: we considered a model converged when the linear trend of the loss curve was negligible and successive steps produced only marginal changes, indicating that further optimisation would not meaningfully alter the parameter estimates.

**Variance decomposition.** To attribute the variance in community composition to ecologically interpretable sources, we applied a Monte Carlo ANOVA decomposition to each fitted jSDM. The approach repeatedly draws from the fitted model with one or more components omitted and quantifies how much variation is lost, yielding seven non-overlapping Venn-diagram fractions:

-   **A** — variance uniquely attributable to the abiotic (environment) component;
-   **S** — variance attributable to the spatial component (MEMs);
-   -   **B** — variance attributable to the residual biotic covariance (co-occurrence patterns unexplained by environment or space, interpreted as a signal of potential biotic interactions ([**Warton2015?**](#ref-Warton2015)));
-   **AB, AS, BS, ABS** — variance jointly attributable to pairs or all three components.

We set fractions that were negative — an artefact of Monte Carlo approximation error — to zero, as negative explained variance has no ecological interpretation.

#### 0.2.3.3 Spatial axis

To isolate the effect of geographic extent on co-occurrence patterns, we applied the shared workflow independently at three nested spatial scales while keeping the temporal and taxonomic dimensions constant. The three scales were chosen to span the range from macroecological processes — where broad climatic gradients dominate co-occurrence — to local processes, where dispersal limitation and microhabitat filtering are more prominent:

-   **Continental** — three units: Europe (−10° to 40°E, 35° to 70°N), North America (−130° to −60°W, 30° to 70°N), and northern Asia (60° to 140°E, 50° to 75°N).
-   **Regional** — approximately 30 geographic units per continent, each spanning roughly 20° longitude × 10°–15° latitude.
-   **Local** — approximately 100 fine-grain units per continent, each spanning roughly 5° longitude × 5° latitude.

We compared variance fractions across scales to identify which components of co-occurrence signal are scale-invariant and which are scale-dependent.

In [ ]:
config_spatial_continental <-
  config::get(
    config = "project_spatial_continental",
    file = here::here("config.yml")
  )

config_spatial_regional <-
  config::get(
    config = "project_spatial_regional",
    file = here::here("config.yml")
  )

config_spatial_local <-
  config::get(
    config = "project_spatial_local",
    file = here::here("config.yml")
  )

min_n_cores <-
  config::get(
    config = "default",
    file = here::here("config.yml")
  )$data_processing$min_n_cores

min_n_samples_continental <-
  config_spatial_continental$data_processing$min_n_samples

min_n_samples_regional <-
  config_spatial_regional$data_processing$min_n_samples

min_n_samples_local <-
  config_spatial_local$data_processing$min_n_samples

min_n_taxa_continental <-
  config_spatial_continental$data_processing$min_n_taxa

min_n_taxa_regional <-
  config_spatial_regional$data_processing$min_n_taxa

min_n_taxa_local <-
  config_spatial_local$data_processing$min_n_taxa

Three scale-specific thresholds governed which spatial units were retained for model fitting. We excluded a taxon from a spatial unit if it appeared in fewer than 5 distinct sediment cores. We further excluded taxa if they appeared in fewer than 10 distinct site–age combinations at the continental and regional scales, or fewer than 5 at the local scale, reflecting the reduced number of sites per unit at finer grains. We excluded a spatial unit entirely if fewer than 5 qualifying taxa remained after all pre-processing at the continental scale; the corresponding lower bounds were 4 taxa at the regional scale and 3 taxa at the local scale.

#### 0.2.3.4 Temporal axis

To test whether the structure of plant co-occurrence has changed through time — and in particular whether the large climate changes since the Last Glacial Maximum are associated with a reorganisation of co-occurrence networks — we applied the shared workflow independently to each 500-year time slice across the full 20,000-year window. We fitted a separate jSDM to each time slice rather than a single pooled model with a temporal random effect. Prior to model fitting, we computed co-occurrence network metrics directly from the binarised community matrix at each time slice to characterise the structural properties of the co-occurrence network independently of any model assumptions. We then aggregated results from successive time slices into temporal trajectories of variance fractions and network metrics, revealing how the relative importance of environment, space, and biotic residuals — and the structure of the co-occurrence network itself — has evolved over geological time. We ran analyses for three continental regions: Europe, North America, and northern Asia.

In [ ]:
config_temporal <-
  config::get(
    config = "project_temporal_europe",
    file = here::here("config.yml")
  )

min_n_samples_temporal <-
  config_temporal$data_processing$min_n_samples

min_n_taxa_temporal <-
  config_temporal$data_processing$min_n_taxa

We applied the same three data-quality thresholds at each time slice within each continental region. We excluded a taxon from a given time slice if it appeared in fewer than 5 distinct sediment cores, or in fewer than 10 distinct site–age combinations. We excluded a time slice entirely if fewer than 5 qualifying taxa remained after all pre-processing.

#### 0.2.3.5 Taxonomic axis

The level at which taxa are defined determines which ecological processes are legible in co-occurrence patterns: genus-level co-occurrence may reflect fine-grained competitive or facilitative interactions, whereas broader groupings may reveal assembly rules driven by shared functional strategies. To test how strongly co-occurrence patterns depend on the level of biological organisation, we collapsed the community matrices to three taxonomic resolutions: **Genus**, **Family** and **Functional type (FT)** — ecologically coherent groups derived from functional trait clustering (see *Functional-type classification* below).

We applied the same jSDM pipeline at each resolution without modification, enabling direct comparison of variance fractions across taxonomic scales. We evaluated the taxonomic axis only at the spatial scale, fitting each resolution independently per spatial unit.

### 0.2.4 Functional-type classification

**Trait data and quality control.** Plant functional traits characterise the ecological strategies of taxa and provide a biologically grounded basis for grouping species into functional types that reflect shared responses to the environment rather than arbitrary taxonomic boundaries. We extracted trait data from VegVault for all available trait domains (see Supplementary Table S\[TODO\] for the full list of traits and units). Because trait databases aggregate measurements from many sources with varying units and methodologies, raw values frequently contain outliers and unit errors that would distort the distance matrix used for clustering. We therefore applied a two-stage quality-control workflow to the trait data, reviewing and correcting all flagged records manually. In the first stage, we identified outlier taxa using an inter-quartile range criterion and corrected or excluded them after review. In the second stage, we applied a further quality-control pass after taxonomic harmonisation to remove trait values assigned to ambiguous or unresolvable morphotypes. We aggregated validated values to taxon-level medians per trait domain and assembled them into a taxon × trait matrix for clustering.

**Distance computation and clustering.** We derived functional types separately for each continental unit (Europe, North America, northern Asia) to ensure that the groupings reflect the functional diversity actually present in each region rather than a globally imposed classification. We quantified pairwise dissimilarities among taxa using the Gower distance metric, which is appropriate for plant trait data because it handles mixed variable types (continuous, binary, ordinal) and missing values natively, without requiring imputation or prior standardisation of trait distributions. We then grouped taxa by Ward’s minimum-variance hierarchical clustering applied to the Gower distance matrix. We chose Ward’s method because it minimises within-cluster total variance at each merge step, producing compact, well-separated clusters that are more likely to correspond to ecologically coherent functional strategies than linkage methods that optimise only local cohesion.

**Selection of the number of functional types.** We determined the number of functional types *k* by a dual criterion designed to balance cluster quality with data availability for downstream modelling. For each candidate *k* in the range *k*<sub>min</sub> (= 5) to *k*<sub>max</sub> (= 30), we passed the resulting FT-level community matrix through the same data-filtering steps (rare-type removal, minimum-site and minimum-sample filters, and binarisation). We considered viable only *k* values that left at least a minimum number of non-constant FT columns after filtering, because a *k* that produces many rare or invariant functional types provides insufficient data for stable model estimation regardless of its clustering quality. Among the viable candidates, we selected the *k* with the highest mean silhouette width — a model-free measure of cluster cohesion and separation that does not depend on the clustering algorithm’s internal objective — as the final number of functional types. This two-stage criterion prevents the selection algorithm from choosing a fine-grained *k* that scores well on clustering metrics but collapses to an inadequate number of informative functional types after real-data filtering. We then assigned each taxon its functional-type label and collapsed the community matrix to FT level by taking the union of presences across all member taxa within each site and time point.

### 0.2.5 Computational reproducibility

Reproducibility of the full analysis was a primary design goal, because the study spans hundreds of independent models across three continents, three spatial scales, and 40 time slices — a scope at which silent failures and undocumented parameter choices are difficult to detect after the fact. We implemented the entire analytical workflow — from data extraction through modelling, variance decomposition, and figure generation — as a declarative, dependency-tracked pipeline using the {targets} R package ([**Landau2021?**](#ref-Landau2021)), which caches the output of every computation and automatically re-runs only those steps that are invalidated by an upstream change in code or data. This design makes every result traceable to the exact code and inputs that produced it and ensures that partial re-runs after code edits produce the same output as a full run from scratch. We locked package versions using {renv} ([**Ushey2023?**](#ref-Ushey2023)), so that the full computational environment — including all R package versions and their dependencies — can be recreated identically on any machine.

In [ ]:
n_functions <-
  base::length(
    base::list.files(
      here::here("R/Functions"),
      pattern = "\\.R$",
      recursive = TRUE
    )
  )

n_tests <-
  base::length(
    base::list.files(
      here::here(
        "R/03_Supplementary_analyses/Testing/testthat"
      ),
      pattern = "\\.R$",
      recursive = TRUE
    )
  )

coverage_pct <-
  jsonlite::read_json(
    here::here(
      "Documentation/Functions_test_coverage/covr_report_summary.json"
    )
  ) |>
  purrr::chuck("value") |>
  base::round(digits = 0)

The analytical codebase comprises 113 modular R functions, each in its own file, spanning data extraction, taxonomic harmonisation, community pre-processing, collinearity screening, spatial eigenvector computation, model fitting, variance decomposition, functional-type classification, and visualisation. Each function has a dedicated unit-test file (112 test files in total), and the test suite achieves 92 % line coverage across all function files as measured by {covr} \[TODO: cite\]. All 113 functions are documented and are publicly accessible through a dedicated project documentation website \[GitHub Pages URL; TODO\], which provides searchable reference pages for every function alongside narrative descriptions of each analysis phase.

All analysis code, pipeline definitions, configuration files, and instructions for reproducing the environment are publicly available at \[GitHub URL / Zenodo DOI; TODO\].

## 0.3 Results

In [ ]:
# Example: read and plot co-occurrence summary target
# res_associations <- targets::tar_read(
#   name = "res_associations",
#   store = set_store
# )

### 0.3.1 Data overview

TODO: Report number of sites, taxa, time windows, and samples retained after filtering (use inline `r` expressions — no hardcoded numbers).

### 0.3.2 Model performance

TODO: Report predictive performance metrics (AUC, RMSE, etc.).

### 0.3.3 Co-occurrence patterns

TODO: Describe spatial and temporal patterns in species associations.

### 0.3.4 Environmental drivers

TODO: Describe ANOVA variance partitioning results.

## 0.4 Discussion

TODO: Write discussion.

## 0.5 Acknowledgements

TODO: Add acknowledgements. Funding sources, data providers, computing resources, field/lab assistance.

## 0.6 Author Contributions

| Role                       | Contributors |
|----------------------------|--------------|
| Conceptualisation          |              |
| Data curation              |              |
| Formal analysis            |              |
| Funding acquisition        |              |
| Investigation              |              |
| Methodology                |              |
| Project administration     |              |
| Resources                  |              |
| Software                   |              |
| Supervision                |              |
| Validation                 |              |
| Visualisation              |              |
| Writing – original draft   |              |
| Writing – review & editing |              |

## 0.7 Data Availability

TODO: Describe data availability. Reference VegVault, pipeline outputs, and any archived datasets.

## 0.8 Code Availability

The analysis code is available at <https://github.com/OndrejMottl/BIODYNAMICS-vegetation-cooccurrence> and archived at TODO (Zenodo DOI to be added upon acceptance).

## 0.9 References

# Supplementary Information

## 0.10 Supplementary Figures

### Figure S1 — TODO title

In [ ]:
# TODO: Add figure code here.

## 0.11 Supplementary Tables

### Table S1 — TODO title

``` r
# TODO: Add table code here.
```

Table 1: TODO: Table S1 caption.

## 0.12 Supplementary Methods

TODO: Add any extended methodological details not included in the main Methods section (e.g., sensitivity analyses, model diagnostics, additional filtering criteria).

```` markdown
---
title: >
  TODO: manuscript title
author:
  - name: Ondřej Mottl
    orcid: 0000-0002-9796-5081
    email: ondrej.mottl@gmail.com
    corresponding: true
    affiliations:
      - id: natur-cuni
        name: >
          Department of Botany, Faculty of Science,
          Charles University
        city: Prague
        country: Czech Republic
  - name: TODO Co-author 2
    orcid: 0000-0000-0000-0000
    affiliations:
      - id: aff2
        name: TODO Institution
        city: TODO City
        country: TODO Country
  - name: TODO Co-author 3
    orcid: 0000-0000-0000-0000
    affiliations:
      - ref: aff2
date: last-modified
date-format: long
abstract: |
  TODO: Write abstract (~150 words for Nature Letter).
  Background sentence(s). Key question. Approach in one sentence.
  Key findings (2–3 sentences). Broader implication (1 sentence).
keywords:
  - vegetation co-occurrence
  - palaeoecology
  - joint species distribution model
  - HMSC
  - Holocene
bibliography: references.bib
# Uncomment the line below to apply a journal citation style:
# csl: https://www.zotero.org/styles/nature
---

quarto-executable-code-5450563D

```r
#| label: setup
#| include: false
library(here)
library(targets)

here::i_am("Documentation/Manuscript/manuscript.qmd")

suppressMessages(
  suppressWarnings(
    source(here::here("R/___setup_project___.R"))
  )
)

# Set active configuration — adjust as needed
Sys.setenv(R_CONFIG_ACTIVE = "project_cz")

# Build the targets store path
# (pattern from website pages; update pipeline name as needed)
vec_pipeline <- "pipeline_basic"

set_store <-
  base::paste0(
    get_active_config("target_store"), "/", vec_pipeline, "/"
  ) |>
  here::here()
```

{{< include sections/02_introduction.qmd >}}

{{< include sections/03_methods.qmd >}}

{{< include sections/04_results.qmd >}}

{{< include sections/05_discussion.qmd >}}

{{< include sections/07_acknowledgements.qmd >}}

{{< include sections/08_author_contributions.qmd >}}

{{< include sections/09_data_availability.qmd >}}

{{< include sections/10_code_availability.qmd >}}

{{< include sections/11_references.qmd >}}

{{< include sections/06_supplementary.qmd >}}
````